In [2]:
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.datasets import load_diabetes

In [3]:
X, y = load_diabetes(return_X_y=True)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=23)

In [5]:
X_train.shape

(353, 10)

In [6]:
lr = LinearRegression()

In [7]:
lr.fit(X_train, y_train)

LinearRegression()

In [8]:
y_pred = lr.predict(X_test)

In [9]:
r2_score(y_test, y_pred)

0.4588389962630465

In [15]:
print(lr.coef_ , lr.intercept_)

[  -19.84569955  -271.44799521   518.60568284   326.13921989
 -1003.87022985   624.04203978   220.7339424    263.93739547
   829.85679767    77.75019971] 151.5282462062113


# Using our own Mini Batch GD class

In [46]:
class MBGDreg:
    def __init__(self, batch_size, learning_rate, epochs):
        self.coef_ = None
        self.intercept_ = None
        self.batch_size = batch_size
        self.lr = learning_rate
        self.epochs = epochs

    def fit(self, X_train, y_train):
        self.intercept_ = 0 
        self.coef_ = np.ones(X_train.shape[1])

        for i in range(self.epochs):
            for j in range(int(X_train.shape[0]/self.batch_size)):
                
                index = random.sample(range(X_train.shape[0]), self.batch_size)
                y_hat = np.dot(X_train[index], self.coef_) + self.intercept_
                intercept_slope = -2 * np.mean(y_train[index] - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_slope)
    
                coef_slope = -2 * np.dot((y_train[index] - y_hat) , X_train[index]) / X_train.shape[0]
                self.coef_ = self.coef_ - (self.lr * coef_slope)
                
        print(self.intercept_, self.coef_)

    def predict(self, x_test):
        return np.dot(X_test, self.coef_) + self.intercept_

In [47]:
#index = random.sample(range(X_train.shape[0]), 10)
#index

In [209]:
mbgd = MBGDreg(batch_size=int(X_train.shape[0]/50), learning_rate=0.6, epochs=300)

In [210]:
mbgd.fit(X_train, y_train)

153.0775547412978 [  33.91258477  -94.84365728  335.07825011  213.72430385   21.12607035
  -13.24277126 -187.30431665  158.45646959  304.47903798  159.16436056]


In [211]:
y_pred = mbgd.predict(X_test)

In [212]:
r2_score(y_test, y_pred)

0.4284064101539674

In [215]:
int(253/50)

5

# Now using sklearn

In [213]:
from sklearn.linear_model import SGDRegressor

In [214]:
sgd = SGDRegressor(learning_rate="constant", eta0=0.6)

In [221]:
batch_size = 30

for i in range(100):
    index = random.sample(range(X_train.shape[0]), batch_size)
    sgd.partial_fit(X_train[index], y_train[index])

In [222]:
sgd.coef_, sgd.intercept_

(array([ -40.09983931, -242.96094037,  513.56897038,  305.21767062,
        -138.72598069,  -65.34762734, -135.56419028,  158.12382657,
         503.94078521,   86.58963528]),
 array([168.8740837]))

In [223]:
y_pred = sgd.predict(X_test)

In [224]:
r2_score(y_test, y_pred)

0.4369643038493589